# ua-legal-lm-ft — дообучение готовой многоязычной LLM на украинских юридических данных

База: **Qwen/Qwen2.5-0.5B** (multilingual, украинский в претрейне). Метод: **LoRA** (peft) на GPU Kaggle.
Данные: метаданные решений ЄДРСР + компании ЄДР + ПДВ + (если уже собраны) полные тексты решений
из [JoTalbot/ua-edrsr-texts](https://huggingface.co/datasets/JoTalbot/ua-edrsr-texts).
Результат публикуется в `JoTalbot/ua-legal-lm-ft` при наличии секрета `HF_TOKEN`.


In [ ]:
# Конфигурация
BASE_MODEL = 'Qwen/Qwen2.5-0.5B'
YEARS = [2024, 2025, 2026]
PARTS_PER_YEAR = 4          # метаданные (по 250k записей)
TEXTS_SHARDS = 8            # шардов полных текстов (если датасет уже существует)
EDR_LIMIT, VAT_LIMIT = 400_000, 150_000
CTX, BATCH, GRAD_ACCUM, STEPS, LR = 512, 2, 4, 2000, 1e-4
LORA_R, LORA_ALPHA = 16, 32
HF_TEXTS = 'JoTalbot/ua-edrsr-texts'
HF_EDRSR = 'JoTalbot/ua-edrsr'
HF_OPEN = 'JoTalbot/ua-open-data'
HF_MODEL_REPO = 'JoTalbot/ua-legal-lm-ft'
REPO = 'https://github.com/JoTalbot/ukraine'
print('config ok')


In [ ]:
# Установка (numpy/torch не трогаем — согласованы с системой; при P100 ниже ставится совместимый torch)
!git clone -q {REPO} || true
!pip -q install tokenizers pyarrow peft striprtf
import os, glob, json, subprocess, urllib.request, pathlib
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), '| capability:', cap)
    if cap[0] < 7:
        subprocess.run(['pip','install','-q','torch==2.7.1','--index-url','https://download.pytorch.org/whl/cu118'], check=True)
        print('Pascal: torch 2.7.1+cu118 установлен (используется subprocess-обучением ниже)')


In [ ]:
# Данные: метаданные + полные тексты (если есть)
def ls(repo, prefix=''):
    url = f'https://huggingface.co/api/datasets/{repo}/tree/main/{prefix}'
    try:
        with urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent':'kaggle'}), timeout=30) as r:
            return [i['path'] for i in json.load(r)]
    except Exception:
        return []
os.makedirs('data/edrsr', exist_ok=True); os.makedirs('data/texts', exist_ok=True)
for y in YEARS:
    for p in sorted(ls(HF_EDRSR, str(y)))[:PARTS_PER_YEAR]:
        dest = 'data/edrsr/' + p.replace('/','_')
        if not os.path.exists(dest):
            urllib.request.urlretrieve(f'https://huggingface.co/datasets/{HF_EDRSR}/resolve/main/{p}', dest)
text_paths = sorted([p for p in ls(HF_TEXTS) if '/texts-' in p])[:TEXTS_SHARDS]
text_files = []
for p in text_paths:
    dest = 'data/texts/' + p.replace('/','_')
    if not os.path.exists(dest):
        try:
            urllib.request.urlretrieve(f'https://huggingface.co/datasets/{HF_TEXTS}/resolve/main/{p}', dest)
            text_files.append(dest)
        except Exception as e:
            print('texts skip', p, e)
    else:
        text_files.append(dest)
urllib.request.urlretrieve(f'https://huggingface.co/datasets/{HF_OPEN}/resolve/main/edr/UO.zip', 'data/UO.zip')
urllib.request.urlretrieve(f'https://huggingface.co/datasets/{HF_OPEN}/resolve/main/vat_payers/pdv_actual.csv', 'data/pdv.csv')
print('метаданных:', len(glob.glob('data/edrsr/*')), '| шардов текстов:', len(text_files))


In [ ]:
# Корпус (метаданные + тексты решений)
cmd = ['python','ukraine/scripts/build_lm_corpus.py',
       '--edr-uo','data/UO.zip','--vat','data/pdv.csv',
       f'--edrsr-limit={500_000}', f'--edr-limit={EDR_LIMIT}', f'--vat-limit={VAT_LIMIT}',
       '--output','data/corpus.txt']
for f in glob.glob('data/edrsr/*'): cmd += ['--edrsr-parquet', f]
for f in text_files: cmd += ['--texts-parquet', f]
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-1000:])
print('corpus bytes:', os.path.getsize('data/corpus.txt'))


In [ ]:
# Обучение LoRA (subprocess — совместимость torch после замены для Pascal; traceback -> model-ft/error.txt)
train_src = r'''
import json, math, os, pathlib, random
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import subprocess as sp
BASE = os.environ['BASE_MODEL']; CTX = int(os.environ['CTX']); STEPS = int(os.environ['STEPS'])
BATCH = int(os.environ['BATCH']); LR = float(os.environ['LR'])
GRAD_ACCUM = int(os.environ.get('GRAD_ACCUM', '4'))
tok = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16)
model = get_peft_model(model, LoraConfig(
    r=int(os.environ['LORA_R']), lora_alpha=int(os.environ['LORA_ALPHA']), lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type='CAUSAL_LM'))
model.print_trainable_parameters()
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
if torch.cuda.is_available(): model = model.cuda()
lines = [l.strip() for l in open('data/corpus.txt', encoding='utf-8') if l.strip()]
random.Random(7).shuffle(lines)
chunks, buf = [], ''
for l in lines:
    buf = (buf + '\n' + l).strip()
    if len(buf) > 1200:
        chunks.append(buf); buf = ''
print('chunks:', len(chunks))
def encode(batch_lines):
    out = tok(batch_lines, truncation=True, max_length=CTX, padding='max_length',
              return_tensors='pt')
    return out['input_ids'], out['attention_mask']
n_val = max(64, len(chunks)//50)
val_chunks, train_chunks = chunks[:n_val], chunks[n_val:]
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=STEPS, pct_start=0.05)
metrics = open('model-ft/metrics.jsonl', 'a', encoding='utf-8')
step = 0; i = 0; g = torch.Generator().manual_seed(7)
def val_loss():
    model.eval(); tot = n = 0
    with torch.no_grad():
        for j in range(0, min(256, len(val_chunks)), BATCH):
            ids, mask = encode(val_chunks[j:j+BATCH])
            if torch.cuda.is_available(): ids, mask = ids.cuda(), mask.cuda()
            out = model(input_ids=ids, attention_mask=mask, labels=ids)
            tot += out.loss.item(); n += 1
    model.train(); return tot / max(n,1)
model.train()
while step < STEPS:
    for _ in range(GRAD_ACCUM):
        if i + BATCH > len(train_chunks): i = 0
        ids, mask = encode(train_chunks[i:i+BATCH]); i += BATCH
        if torch.cuda.is_available(): ids, mask = ids.cuda(), mask.cuda()
        out = model(input_ids=ids, attention_mask=mask, labels=ids)
        (out.loss / GRAD_ACCUM).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step(); opt.zero_grad()
    step += 1
    if step % 25 == 0:
        m = {'step': step, 'loss': round(out.loss.item(), 4)}
        if step % 250 == 0:
            m['val_loss'] = round(val_loss(), 4)
            ids0, _ = encode(['Суд: '])
            ids0 = ids0[:, :4]
            model.eval()
            with torch.no_grad():
                gen = model.generate(input_ids=ids0.cuda() if torch.cuda.is_available() else ids0,
                                     max_new_tokens=80, do_sample=True, temperature=0.8,
                                     pad_token_id=tok.eos_token_id)
            model.train()
            with open('model-ft/samples.txt', 'a', encoding='utf-8') as f:
                f.write(f'\n===== step {step} =====\n' + tok.decode(gen[0], skip_special_tokens=True) + '\n')
        metrics.write(json.dumps(m) + '\n'); metrics.flush()
        print(m, flush=True)
model.eval()
merged = model.merge_and_unload()
merged.save_pretrained('model-ft/final', safe_serialization=True)
tok.save_pretrained('model-ft/final')
print('merged model saved')
'''
pathlib.Path('train_ft.py').write_text(train_src, encoding='utf-8')
env = dict(os.environ, BASE_MODEL=BASE_MODEL, CTX=str(CTX), STEPS=str(STEPS), BATCH=str(BATCH),
           LR=str(LR), LORA_R=str(LORA_R), LORA_ALPHA=str(LORA_ALPHA), GRAD_ACCUM=str(GRAD_ACCUM))
os.makedirs('model-ft', exist_ok=True)
r = subprocess.run(['python','train_ft.py'], env=env, capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-3000:])
pathlib.Path('model-ft/error.txt').write_text('ok' if r.returncode == 0 else (r.stderr[-20000:] or 'no stderr'), encoding='utf-8')
print('FT exit code:', r.returncode)


In [ ]:
# Публикация в HF Hub (если в Kaggle добавлен секрет HF_TOKEN)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')
if token and os.path.isdir('model-ft/final'):
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    api.create_repo(repo_id=HF_MODEL_REPO, repo_type='model', exist_ok=True)
    api.upload_folder(folder_path='model-ft/final', repo_id=HF_MODEL_REPO, repo_type='model',
                      commit_message='ua-legal-lm-ft LoRA merge')
    for f in ['model-ft/metrics.jsonl','model-ft/samples.txt']:
        if os.path.exists(f): api.upload_file(path_or_fileobj=f, path_in_repo=f.split('/')[-1],
                                               repo_id=HF_MODEL_REPO, repo_type='model')
    print('published to', HF_MODEL_REPO)
else:
    print('HF_TOKEN не задан или модель не обучена — результат в /kaggle/working/model-ft')


In [ ]:
# Очистка тяжёлых файлов (output = /kaggle/working)
import shutil
for p in ['data', 'ukraine']:
    if os.path.exists(p): shutil.rmtree(p)
print('remaining:', os.listdir('.'), os.listdir('model-ft') if os.path.isdir('model-ft') else [])


## Замечания
- T4/P100 ~16 ГБ: 0.5B fp16 + LoRA; batch 2×512 × накопление 4 (эффективный batch 8)
  + gradient checkpointing — защита от OOM при расчёте loss (logits vocab 151936 в fp32).
- ~2000 шагов ≈ 40–70 мин на P100.
- Если датасет полных текстов ещё пуст, обучение идёт на метаданных+ЄДР (как GPU-from-scratch).
